In [0]:
# Charger la table Gold précédente
from pyspark.sql.functions import (
    ceil,
    col,
    current_date,
    current_timestamp,
    date_add,
    least,
    lit,
    when
)

inventory_health = spark.table(
    "retail_dev.gold.inventory_health"
)

In [0]:
# Calculons le score de risque 
stockout_risk = (
    inventory_health

    # Risque lié au niveau du stock
    .withColumn(
        "stock_position_score",
        when(
            col("stock_status") == "OUT_OF_STOCK",
            70
        )
        .when(
            col("stock_status") == "LOW_STOCK",
            50
        )
        .when(
            col("estimated_quantity_on_hand")
            < col("target_stock_level"),
            20
        )
        .otherwise(0)
    )

    # Risque lié au nombre de jours de couverture
    .withColumn(
        "coverage_score",
        when(col("days_of_cover") <= 3, 20)
        .when(col("days_of_cover") <= 7, 15)
        .when(col("days_of_cover") <= 14, 10)
        .when(col("days_of_cover") <= 30, 5)
        .otherwise(0)
    )

    # Risque lié à la vitesse des ventes
    .withColumn(
        "sales_velocity_score",
        when(col("average_daily_sales") >= 5, 10)
        .when(col("average_daily_sales") >= 2, 5)
        .otherwise(0)
    )

    # Score final limité à 100
    .withColumn(
        "risk_score",
        least(
            lit(100),
            col("stock_position_score")
            + col("coverage_score")
            + col("sales_velocity_score")
        )
    )
)

In [0]:
# Classer les risques et recommander une action
stockout_risk = (
    stockout_risk

    .withColumn(
        "risk_level",
        when(col("risk_score") >= 70, "CRITICAL")
        .when(col("risk_score") >= 50, "HIGH")
        .when(col("risk_score") >= 25, "MEDIUM")
        .otherwise("LOW")
    )

    .withColumn(
        "estimated_stockout_date",
        when(
            col("estimated_quantity_on_hand") <= 0,
            current_date()
        )
        .when(
            col("days_of_cover").isNotNull(),
            date_add(
                current_date(),
                ceil(col("days_of_cover")).cast("int")
            )
        )
        .otherwise(lit(None).cast("date"))
    )

    .withColumn(
        "recommended_action",
        when(
            col("stock_status") == "OUT_OF_STOCK",
            "Réapprovisionnement urgent"
        )
        .when(
            col("stock_status") == "LOW_STOCK",
            "Commander immédiatement"
        )
        .when(
            col("stock_status") == "OVERSTOCK",
            "Réduire les achats ou lancer une promotion"
        )
        .when(
            col("risk_level") == "MEDIUM",
            "Surveiller et planifier une commande"
        )
        .otherwise("Aucune action urgente")
    )

    .withColumn(
        "_gold_processed_at",
        current_timestamp()
    )
)

In [0]:
# Enregistrer la table 
(
    stockout_risk.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_dev.gold.stockout_risk"
    )
)

print(
    "Table créée : "
    "retail_dev.gold.stockout_risk"
)

In [0]:
# Afficher les produits prioritaires
display(
    spark.table("retail_dev.gold.stockout_risk")
    .select(
        "stock_item_id",
        "stock_item_name",
        "supplier_name",
        "estimated_quantity_on_hand",
        "reorder_level",
        "average_daily_sales",
        "days_of_cover",
        "risk_score",
        "risk_level",
        "recommended_reorder_quantity",
        "recommended_action"
    )
    .orderBy(
        col("risk_score").desc(),
        col("estimated_quantity_on_hand").asc()
    )
)

In [0]:
# Résumé des risques
from pyspark.sql.functions import count, sum as spark_sum

display(
    spark.table("retail_dev.gold.stockout_risk")
    .groupBy("risk_level")
    .agg(
        count("*").alias("product_count"),
        spark_sum("recommended_reorder_quantity")
        .alias("units_to_reorder")
    )
    .orderBy(col("risk_level"))
)